# Gold Price Pipeline Validation

این notebook فقط برای **validation** چهار لایه‌ی pipeline است:

1. Bronze
2. Silver
3. Gold 1-minute bars
4. Gold metrics

این notebook برای **inspect و check** است، نه برای اجرای کپی‌شده‌ی jobها.


## 1) Spark / Delta / MinIO setup

اگر از قبل یک `SparkSession` فعال داری، این cell آن را می‌بندد و یک session جدید برای خواندن Delta روی MinIO می‌سازد.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

PACKAGES = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

builder = (
    SparkSession.builder
    .appName("gold-pipeline-validation")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark


## 2) Table paths

در صورت نیاز فقط همین cell را ویرایش کن.


In [ ]:
BRONZE_PATH = "s3a://lakehouse/bronze/gold_price_events"
SILVER_PATH = "s3a://lakehouse/silver/gold_price_ticks_clean"
GOLD_BARS_PATH = "s3a://lakehouse/gold/gold_price_bars_1m"
METRICS_PATH = "s3a://lakehouse/gold/gold_price_metrics"

print(BRONZE_PATH)
print(SILVER_PATH)
print(GOLD_BARS_PATH)
print(METRICS_PATH)


## 3) Helper functions

In [ ]:
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, when

def read_delta(path):
    return spark.read.format("delta").load(path)

def show_basic_info(name, df, time_col=None, n=10):
    print(f"===== {name} =====")
    print("row_count =", df.count())
    df.printSchema()
    if time_col:
        df.orderBy(col(time_col).desc()).show(n, truncate=False)
    else:
        df.show(n, truncate=False)

def duplicate_count(df, key_col):
    agg = df.agg(
        count("*").alias("rows"),
        countDistinct(key_col).alias("distinct_keys")
    ).collect()[0]
    return agg["rows"] - agg["distinct_keys"]


## 4) Bronze validation

چک‌های اصلی:
- وجود داده
- پر بودن `event_id`
- وضعیت `api_status`
- پر بودن `source_event_ts` و `price_usd` در رکوردهای OK


In [ ]:
bronze_df = read_delta(BRONZE_PATH)
show_basic_info("BRONZE", bronze_df, time_col="ingestion_ts")

bronze_checks = bronze_df.agg(
    count("*").alias("rows"),
    countDistinct("event_id").alias("distinct_event_id"),
    spark_sum(when(col("event_id").isNull(), 1).otherwise(0)).alias("null_event_id"),
    spark_sum(when(col("api_status") == "OK", 1).otherwise(0)).alias("ok_rows"),
    spark_sum(when((col("api_status") == "OK") & col("source_event_ts").isNull(), 1).otherwise(0)).alias("ok_with_null_source_event_ts"),
    spark_sum(when((col("api_status") == "OK") & col("price_usd").isNull(), 1).otherwise(0)).alias("ok_with_null_price"),
).collect()[0]

print(dict(bronze_checks.asDict()))
print("duplicate_event_id =", duplicate_count(bronze_df, "event_id"))


## 5) Silver validation

چک‌های اصلی:
- duplicate نداشتن بر اساس `event_id`
- فقط رکورد valid
- پر بودن `event_ts_utc` و `price_usd`


In [ ]:
silver_df = read_delta(SILVER_PATH)
show_basic_info("SILVER", silver_df, time_col="ingestion_ts")

silver_checks = silver_df.agg(
    count("*").alias("rows"),
    countDistinct("event_id").alias("distinct_event_id"),
    spark_sum(when(col("event_ts_utc").isNull(), 1).otherwise(0)).alias("null_event_ts_utc"),
    spark_sum(when(col("price_usd").isNull(), 1).otherwise(0)).alias("null_price_usd"),
    spark_sum(when(col("price_usd") <= 0, 1).otherwise(0)).alias("non_positive_price"),
    spark_sum(when(col("quality_flag") != "OK", 1).otherwise(0)).alias("non_ok_quality_flag"),
).collect()[0]

print(dict(silver_checks.asDict()))
print("duplicate_event_id =", duplicate_count(silver_df, "event_id"))


## 6) Gold 1-minute bars validation

چک‌های اصلی:
- پر بودن OHLC
- منطقی بودن high/low
- مثبت بودن `tick_count`


In [ ]:
gold_bars_df = read_delta(GOLD_BARS_PATH)
show_basic_info("GOLD_BARS_1M", gold_bars_df, time_col="bar_start_ts")

gold_bars_checks = gold_bars_df.agg(
    count("*").alias("rows"),
    spark_sum(when(col("open_price").isNull(), 1).otherwise(0)).alias("null_open"),
    spark_sum(when(col("high_price").isNull(), 1).otherwise(0)).alias("null_high"),
    spark_sum(when(col("low_price").isNull(), 1).otherwise(0)).alias("null_low"),
    spark_sum(when(col("close_price").isNull(), 1).otherwise(0)).alias("null_close"),
    spark_sum(when(col("tick_count") <= 0, 1).otherwise(0)).alias("non_positive_tick_count"),
    spark_sum(when(col("high_price") < col("low_price"), 1).otherwise(0)).alias("high_lt_low"),
    spark_sum(when(col("open_price") > col("high_price"), 1).otherwise(0)).alias("open_gt_high"),
    spark_sum(when(col("open_price") < col("low_price"), 1).otherwise(0)).alias("open_lt_low"),
    spark_sum(when(col("close_price") > col("high_price"), 1).otherwise(0)).alias("close_gt_high"),
    spark_sum(when(col("close_price") < col("low_price"), 1).otherwise(0)).alias("close_lt_low"),
).collect()[0]

print(dict(gold_bars_checks.asDict()))


## 7) Metrics validation

چک‌های اصلی:
- وجود metric rows
- پر بودن `metric_ts`
- تولید شدن moving averages و returns برای ردیف‌های متأخر
- شمارش anomalyها


In [ ]:
metrics_df = read_delta(METRICS_PATH)
show_basic_info("GOLD_METRICS", metrics_df, time_col="metric_ts")

metrics_checks = metrics_df.agg(
    count("*").alias("rows"),
    spark_sum(when(col("metric_ts").isNull(), 1).otherwise(0)).alias("null_metric_ts"),
    spark_sum(when(col("price_usd").isNull(), 1).otherwise(0)).alias("null_price_usd"),
    spark_sum(when(col("return_1m").isNull(), 1).otherwise(0)).alias("null_return_1m"),
    spark_sum(when(col("ma_5").isNull(), 1).otherwise(0)).alias("null_ma_5"),
    spark_sum(when(col("ma_15").isNull(), 1).otherwise(0)).alias("null_ma_15"),
    spark_sum(when(col("ma_60").isNull(), 1).otherwise(0)).alias("null_ma_60"),
    spark_sum(when(col("is_anomaly") == True, 1).otherwise(0)).alias("anomaly_count"),
).collect()[0]

print(dict(metrics_checks.asDict()))


## 8) Pipeline summary

این cell فقط یک خلاصه مدیریتی می‌دهد تا سریع بفهمی pipeline در چه وضعی است.


In [ ]:
summary = {
    "bronze_rows": bronze_df.count(),
    "silver_rows": silver_df.count(),
    "gold_bars_rows": gold_bars_df.count(),
    "metrics_rows": metrics_df.count(),
    "bronze_duplicate_event_id": duplicate_count(bronze_df, "event_id"),
    "silver_duplicate_event_id": duplicate_count(silver_df, "event_id"),
}

for k, v in summary.items():
    print(f"{k}: {v}")


## 9) Optional spot checks

در صورت نیاز این cellها را برای drill-down استفاده کن.


In [ ]:
# bronze_df.filter(col("api_status") != "OK").show(truncate=False)
# silver_df.filter(col("quality_flag") != "OK").show(truncate=False)
# gold_bars_df.filter(col("tick_count") <= 0).show(truncate=False)
# metrics_df.filter(col("is_anomaly") == True).orderBy(col("metric_ts").desc()).show(truncate=False)
